In [13]:
import sys
import random
import gc, argparse
import copy
import time
import glfw
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from PIL import Image
import numpy as np
from datetime import datetime
import json
import os
from src.env.env_clr import RILAB_OMY_ENV
from src.controllers import load_controller 

In [7]:
# Load environment configuration
config_file_path = './configs/train_key_clr.json'
with open(config_file_path) as f:
    env_conf = json.load(f)
language_instruction = env_conf['language_instruction']
omy_env = RILAB_OMY_ENV(cfg=env_conf,
                        seed=None, 
                        action_type=env_conf['control_mode'], 
                        obs_type='eef_pose',
                        vis_mode = 'keyboard',
                        build_mjcf=False)


omy_env.reset(leader_pose = True)
# Load keyboard controller
controller = load_controller('keyboard',env_conf)
controller.reset(omy_env)


-----------------------------------------------------------------------------
name:[tabletop_env] dt:[0.002] HZ:[500]
 n_qpos:[40] n_qvel:[39] n_qacc:[39] n_ctrl:[9]
 integrator:[IMPLICITFAST]

n_body:[35]
 [0/35] [world] mass:[0.00]kg
 [1/35] [vention_rail_carriage] mass:[22.53]kg
 [2/35] [ewellix_lift_higher_link] mass:[19.17]kg
 [3/35] [ewellix_lift_middle_link] mass:[15.59]kg
 [4/35] [shoulder_link] mass:[7.37]kg
 [5/35] [upper_arm_link] mass:[13.05]kg
 [6/35] [forearm_link] mass:[3.99]kg
 [7/35] [wrist_1_link] mass:[2.10]kg
 [8/35] [wrist_2_link] mass:[1.98]kg
 [9/35] [wrist_3_link] mass:[1.56]kg
 [10/35] [finger_1_link] mass:[0.05]kg
 [11/35] [finger_2_link] mass:[0.05]kg
 [12/35] [door] mass:[0.30]kg
 [13/35] [right_latch_pull] mass:[0.10]kg
 [14/35] [left_latch_pull] mass:[0.10]kg
 [15/35] [latch_lock] mass:[0.10]kg
 [16/35] [lorge/hatch_face] mass:[19.28]kg
 [17/35] [lorge/external_rotary_wheel] mass:[0.41]kg
 [18/35] [lorge/external_rotary_handle] mass:[0.03]kg
 [19/35] [lor

You can teleop your robot with keyboard
```
---------     -----------------------
   w       ->        backward
s  a  d        left   forward   right
---------      -----------------------
In x, y plane

---------
R: Moving Up
F: Moving Down
---------
In z axis

---------
Q: Tilt left
E: Tilt right
UP: Look Upward
Down: Look Donward
Right: Turn right
Left: Turn left
---------
For rotation

---------
SPACEBAR: Toggle Gripper
--------

---------
z: reset
--------
```

In [8]:
NUM_TRIALS_PER_TASK = 3
RESUME = False  # Set to True to resume recording into an existing dataset
DATASET_ROOT="./dataset/teleoperation_clr_dataset" #@param {type:"string"}


In [9]:
from src.dataset.utils import make_teleoperation_dataset

if os.path.exists(DATASET_ROOT):
    if RESUME:
        print("RESUME existing dataset")
        dataset = LeRobotDataset('temp', root=DATASET_ROOT)
        print(f"Loaded dataset with {dataset.num_episodes} existing episodes")
    else:
        import shutil
        print("REMOVE")
        shutil.rmtree(DATASET_ROOT)
        print("CREATE")
        dataset = make_teleoperation_dataset(DATASET_ROOT, state_dim=7)
else:
    print("CREATE")
    dataset = make_teleoperation_dataset(DATASET_ROOT, state_dim=7)

REMOVE
CREATE


In [10]:
episode_id = dataset.num_episodes if RESUME else 0
start_episode_id = episode_id
print(f"Starting from episode {episode_id}")

Starting from episode 0


In [11]:
while omy_env.env.is_viewer_alive() and episode_id < start_episode_id + NUM_TRIALS_PER_TASK:
    omy_env.step_env()
    if omy_env.env.loop_every(HZ=20):
        key_list = omy_env.env.get_key_pressed_list()
        done = omy_env.check_success()
        if done or 90 in key_list:  # 'z' key to reset
            print("END EPISODE")
            if done:
                dataset.save_episode()
                episode_id += 1
            else: 
                dataset.clear_episode_buffer()
                #pass
            omy_env.reset(leader_pose = True)
            action = controller.get_action()
            eef_pose = omy_env.step(action)
        action = controller.get_action()
        eef_pose = omy_env.step(action)
        
        agent_image, wrist_image, _, _ = omy_env.grab_image()
        # # # resize to 256x256
        agent_image = Image.fromarray(agent_image)
        wrist_image = Image.fromarray(wrist_image)
        agent_image = agent_image.resize((256, 256))
        wrist_image = wrist_image.resize((256, 256))
        agent_image = np.array(agent_image)
        wrist_image = np.array(wrist_image)
        
        obj_states, recp_q_poses = omy_env.get_object_pose(pad=10)
        obj_poses = np.array(obj_states['poses'])
        
        # Add frame to the dataset
        dataset.add_frame( {
                "observation.image": agent_image,
                "observation.wrist_image": wrist_image,
                "observation.state": eef_pose,
                "action": action,
                "observation.eef_pose": eef_pose,
                'env.obj_pose': np.array(obj_states['poses'],dtype=np.float32),
                "env.obj_names": ','.join(obj_states['names']),
                "env.obj_q_names": ','.join(recp_q_poses['names']),
                "env.obj_q_states": np.array(recp_q_poses['poses'],dtype=np.float32),
                "env.config_file_name": config_file_path,
                "task": language_instruction
            }, 
        )


        last_obj_poses = obj_poses
        # based on the episode_id number, get the guide line

        omy_env.render(language_instruction, guideline= f' [Num Episode: {episode_id}/{NUM_TRIALS_PER_TASK}]')
    omy_env.env.sync_sim_wall_time()
omy_env.env.close_viewer()
dataset.finalize()

END EPISODE


Map: 100%|██████████| 1297/1297 [00:00<00:00, 1507.21 examples/s]


DONE INITIALIZATION
END EPISODE
DONE INITIALIZATION
END EPISODE


Map: 100%|██████████| 852/852 [00:00<00:00, 1515.23 examples/s]


DONE INITIALIZATION
END EPISODE
DONE INITIALIZATION
END EPISODE
DONE INITIALIZATION
END EPISODE
DONE INITIALIZATION
END EPISODE
DONE INITIALIZATION
END EPISODE
DONE INITIALIZATION
END EPISODE


Map: 100%|██████████| 1413/1413 [00:01<00:00, 1387.83 examples/s]


DONE INITIALIZATION


In [14]:
DATASET_REPO="gimarchetti/clr-experiment-dataset" #@param {type:"string"}
POLICY_REPO="gimarchetti/clr-experiment-pi05" #@param {type:"string"}
OUTPUT_DIR="./ckpt/clr-experiment-pi05" #@param {type:"string"}
JOB_NAME="clr-experiment-pi05"+datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
MAX_TRAIN_STEPS=400 #@param {type:"integer"}
CHUNK_SIZE=100 #@param {type:"integer"}
ACTION_STEPS=50 #@param {type:"integer"}
#EVAL_STEPS=1000
#SAVE_STEPS=1000
BATCH_SIZE=32 #@param {type:"integer"}
#LEARNING_RATE=5e-5
#WEIGHT_DECAY=0.01
#WARMUP_STEPS=500
#LOGGING_STEPS=100

In [15]:
# Refresh dataset with latest changes
!hf upload {DATASET_REPO} {DATASET_ROOT}  --repo-type=dataset

Start hashing 7 files.
Finished hashing 7 files.
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...hunk-000/file-000.parquet:   1%|              |  531kB / 97.6MB            

Processing Files (0 / 1)      :   1%|              |  531kB / 97.8MB, 1.33MB/s  
New Data Upload               :   1%|              |  530kB / 97.5MB, 1.33MB/s  


  ...e-000003/frame-000000.png:   2%|▏             |   366B / 21.1kB            



  ...e-000003/frame-000000.png:   2%|▏             |   384B / 22.1kB            




  ...hunk-000/file-000.parquet:   2%|▏             | 1.19kB / 68.4kB            





  ...ataset/meta/tasks.parquet:   2%|▏             |  42.0B / 2.42kB            

  ...hunk-000/file-000.parquet:   1%|▏             | 1.06MB / 97.6MB            


  ...e-000003/frame-000000.png:   2%|▏             |   366B / 21.1kB            



  ...e-000003/frame-000000.png:   2%|▏

## In case of accidental deletion of the dataset
You can download the existing dataset

In [ ]:
!hf download {DATASET_REPO} --repo-type dataset --local-dir {DATASET_ROOT}